# Stock Financial Health Analyzer

## Module Mini Assignment - Interactive Data Analysis Tool

**Track Chosen:** Track 4 – Interactive Data Analysis Tool  
**Product Type:** Streamlit financial dashboard with supporting Python notebook  

### Analytical Problem
Investors, business students, and general users often need a simple way to review both recent stock price performance and basic financial health indicators for a listed company. However, many financial platforms are either too complex or focus only on isolated indicators.

This project aims to build a small Python-based data product that combines stock trend analysis with selected financial health metrics in a user-friendly format.

### Intended User / Audience
The intended users are:
- business and finance students,
- beginner investors,
- users who want a quick exploratory overview of a company’s recent market and financial condition.

### Product Goal
The goal is to provide an interactive tool that:
- retrieves stock market data,
- displays selected financial indicators,
- applies a simple rule-based scoring framework,
- generates an interpretable summary,
- and allows results to be exported.

## 1. Data Source and Project Scope

### Data Sources
This project uses:
- **Yahoo Finance** via the `yfinance` Python package for live stock data and financial information
- **Local CSV files** in the `data/` folder as fallback sources when online retrieval fails

### Why This Dataset Was Selected
Yahoo Finance provides business-relevant financial and market data suitable for educational analysis. It supports:
- historical stock prices,
- company information,
- selected financial indicators.

This makes it appropriate for a small business-related data product.

### Data Access Date
Please update this date before submission.

**Data accessed on:** 2026-04-13

In [68]:
import os
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from IPython.display import display

## 2. Input Parameters

In the Streamlit application, the user enters the ticker symbol and selects the analysis period interactively.

In this notebook, the same parameters are defined manually to reproduce the analytical workflow.

In [69]:
ticker = "AAPL"
period = "6mo"

## 3. Helper Functions

The following functions support:
- data formatting,
- interpretation of financial indicators,
- score generation,
- summary generation,
- and local data filtering.

These functions are consistent with the Streamlit app logic.

In [70]:
def format_value(value, percentage=False):
    if value is None or value == "N/A":
        return "N/A"
    if isinstance(value, (int, float)):
        if percentage:
            return f"{value * 100:.2f}%"
        if value >= 1e12:
            return f"{value / 1e12:.2f}T"
        elif value >= 1e9:
            return f"{value / 1e9:.2f}B"
        elif value >= 1e6:
            return f"{value / 1e6:.2f}M"
        return f"{value:,.2f}"
    return str(value)


def evaluate_liquidity(current_ratio):
    if not isinstance(current_ratio, (int, float)):
        return "Liquidity data is unavailable."
    if current_ratio >= 1.5:
        return "Good liquidity position."
    elif current_ratio >= 1:
        return "Acceptable liquidity position."
    else:
        return "Weak liquidity position."


def evaluate_solvency(debt_to_equity):
    if not isinstance(debt_to_equity, (int, float)):
        return "Solvency data is unavailable."
    if debt_to_equity < 50:
        return "Low leverage and solid solvency."
    elif debt_to_equity <= 150:
        return "Moderate leverage."
    else:
        return "High leverage may indicate higher solvency risk."


def evaluate_profitability(profit_margins):
    if not isinstance(profit_margins, (int, float)):
        return "Profitability data is unavailable."
    if profit_margins > 0.2:
        return "Company shows strong profitability."
    elif profit_margins > 0.1:
        return "Company shows decent profitability."
    else:
        return "Profitability may be weak."


def evaluate_growth(revenue_growth):
    if not isinstance(revenue_growth, (int, float)):
        return "Growth data is unavailable."
    if revenue_growth > 0.1:
        return "Revenue is growing strongly."
    elif revenue_growth > 0:
        return "Revenue is growing."
    else:
        return "Revenue growth is weak or negative."


def score_liquidity(current_ratio):
    if not isinstance(current_ratio, (int, float)):
        return None
    if current_ratio >= 1.5:
        return 25
    elif current_ratio >= 1:
        return 18
    else:
        return 10


def score_solvency(debt_to_equity):
    if not isinstance(debt_to_equity, (int, float)):
        return None
    if debt_to_equity < 50:
        return 25
    elif debt_to_equity <= 150:
        return 18
    else:
        return 10


def score_profitability(profit_margins):
    if not isinstance(profit_margins, (int, float)):
        return None
    if profit_margins > 0.2:
        return 25
    elif profit_margins > 0.1:
        return 18
    else:
        return 10


def score_growth(revenue_growth):
    if not isinstance(revenue_growth, (int, float)):
        return None
    if revenue_growth > 0.15:
        return 25
    elif revenue_growth > 0:
        return 18
    else:
        return 10


def calculate_overall_score(current_ratio, debt_to_equity, profit_margins, revenue_growth):
    scores = {
        "Liquidity": score_liquidity(current_ratio),
        "Solvency": score_solvency(debt_to_equity),
        "Profitability": score_profitability(profit_margins),
        "Growth": score_growth(revenue_growth)
    }

    available_scores = [v for v in scores.values() if v is not None]

    if not available_scores:
        return None, scores

    overall_score = sum(available_scores)
    return overall_score, scores


def score_label(score):
    if score is None:
        return "Unavailable"
    if score >= 85:
        return "Excellent"
    elif score >= 70:
        return "Good"
    elif score >= 50:
        return "Moderate"
    else:
        return "Weak"


def build_overall_summary(liquidity_msg, solvency_msg, profitability_msg, growth_msg, score):
    summary_parts = []

    if score is not None:
        summary_parts.append(
            f"The company receives an overall financial health score of {score}/100, "
            f"which is classified as {score_label(score).lower()}."
        )

    for msg in [liquidity_msg, solvency_msg, profitability_msg, growth_msg]:
        if "unavailable" not in msg.lower():
            summary_parts.append(msg)

    if not summary_parts:
        return (
            "Overall summary: insufficient financial data is available to generate "
            "a meaningful assessment."
        )

    return "Overall summary: " + " ".join(summary_parts)


def period_to_days(period):
    mapping = {
        "1mo": 30,
        "3mo": 90,
        "6mo": 180,
        "1y": 365
    }
    return mapping.get(period, 180)

## 4. Data Loading Functions

A dual-source design is used in this project:
1. retrieve data from Yahoo Finance if available,
2. otherwise use local fallback files.

This makes the product more robust and easier to demonstrate.

In [71]:
def load_stock_history_online(ticker, period):
    stock = yf.Ticker(ticker)
    hist = stock.history(period=period)
    return hist


def load_stock_info_online(ticker):
    stock = yf.Ticker(ticker)
    return stock.info


def load_local_csv_data(ticker, period):
    file_path = os.path.join("data", f"{ticker}.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Local CSV file not found: {file_path}")

    df = pd.read_csv(file_path)

    if "Date" not in df.columns:
        raise ValueError("Local CSV must contain a 'Date' column.")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")
    df = df.set_index("Date")

    if "Close" not in df.columns:
        raise ValueError("Local CSV must contain a 'Close' column.")

    days = period_to_days(period)
    latest_date = df.index.max()
    cutoff_date = latest_date - pd.Timedelta(days=days)
    filtered_df = df[df.index >= cutoff_date]

    return filtered_df


def load_local_financial_info(ticker):
    file_path = os.path.join("data", "financial_info.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Local financial info file not found: {file_path}")

    df = pd.read_csv(file_path)

    if "ticker" not in df.columns:
        raise ValueError("financial_info.csv must contain a 'ticker' column.")

    row = df[df["ticker"].str.upper() == ticker.upper()]

    if row.empty:
        raise ValueError(f"No local financial info found for ticker: {ticker}")

    record = row.iloc[0].to_dict()

    numeric_fields = [
        "marketCap",
        "trailingPE",
        "revenueGrowth",
        "profitMargins",
        "currentRatio",
        "debtToEquity"
    ]

    for field in numeric_fields:
        if field in record:
            try:
                record[field] = float(record[field])
            except:
                record[field] = "N/A"

    return record

## 5. Load and Inspect Stock Price Data

This section attempts to retrieve recent stock price history and records the source used.

In [72]:
hist = None
price_source = None

try:
    hist = load_stock_history_online(ticker, period)
    if hist is not None and not hist.empty:
        price_source = "Yahoo Finance"
except Exception as e:
    print("Online price retrieval failed:", e)

if hist is None or hist.empty:
    try:
        hist = load_local_csv_data(ticker, period)
        if hist is not None and not hist.empty:
            price_source = "Local CSV"
    except Exception as e:
        print("Local CSV price retrieval failed:", e)

print("Price data source:", price_source)
display(hist.head())

Price data source: Yahoo Finance


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-10-13 00:00:00-04:00,248.905587,249.214995,245.092847,247.188858,38142900,0.0,0.0
2025-10-14 00:00:00-04:00,246.130873,248.376592,244.234478,247.298645,35478000,0.0,0.0
2025-10-15 00:00:00-04:00,249.015370,251.340939,246.999209,248.865646,33893600,0.0,0.0
2025-10-16 00:00:00-04:00,247.777729,248.566220,244.663670,246.979248,39777000,0.0,0.0
2025-10-17 00:00:00-04:00,247.548162,252.897966,246.799589,251.810028,49147000,0.0,0.0


### Data Preparation Notes

The retrieved stock data are:
- sorted by date,
- indexed by date,
- filtered according to the selected period,
- and later used to calculate price change and a 20-day moving average.

For this small data product, only a limited amount of preparation is required because the Yahoo Finance structure is already relatively clean.
Numeric fields in the local financial information file are converted to float where possible, and invalid values are replaced with `"N/A"`.

## 6. Load and Inspect Financial Information

In [73]:
info = {}
info_source = None

try:
    info = load_stock_info_online(ticker)
    if isinstance(info, dict) and len(info) > 0:
        info_source = "Yahoo Finance"
except Exception as e:
    print("Online financial info retrieval failed:", e)

if not info or not isinstance(info, dict):
    try:
        info = load_local_financial_info(ticker)
        info_source = "Local financial_info.csv"
    except Exception as e:
        print("Local financial info retrieval failed:", e)
        info = {
            "longName": ticker,
            "sector": "N/A",
            "industry": "N/A",
            "marketCap": "N/A",
            "trailingPE": "N/A",
            "revenueGrowth": "N/A",
            "profitMargins": "N/A",
            "currentRatio": "N/A",
            "debtToEquity": "N/A",
        }
        info_source = "Unavailable"

print("Financial info source:", info_source)
info

Financial info source: Yahoo Finance


{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

## 7. Market Overview Analysis

This part calculates the recent market performance of the selected stock:
- latest closing price,
- total price change,
- percentage change,
- average trading volume.

In [74]:
latest_close = hist["Close"].iloc[-1]
first_close = hist["Close"].iloc[0]
price_change = latest_close - first_close
price_change_pct = (price_change / first_close) * 100 if first_close != 0 else 0
avg_volume = hist["Volume"].mean() if "Volume" in hist.columns else None

market_overview_df = pd.DataFrame({
    "Metric": ["Latest Close Price", "Price Change", "Price Change %", "Average Volume"],
    "Value": [
        f"{latest_close:,.2f} USD",
        f"{price_change:,.2f} USD",
        f"{price_change_pct:.2f}%",
        format_value(avg_volume)
    ]
})

display(market_overview_df)

,Metric,Value
0,Latest Close Price,257.83 USD
1,Price Change,10.64 USD
2,Price Change %,4.30%
3,Average Volume,45.94M


## 8. Recent Stock Data Preview

In [75]:
display(hist.tail())

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-04-07 00:00:00-04:00,256.160004,256.200012,245.699997,253.500000,62148000,0.0,0.0
2026-04-08 00:00:00-04:00,258.450012,259.750000,256.529999,258.899994,41032800,0.0,0.0
2026-04-09 00:00:00-04:00,259.000000,261.119995,256.070007,260.489990,28121600,0.0,0.0
2026-04-10 00:00:00-04:00,259.980011,262.190002,259.019989,260.480011,31259500,0.0,0.0
2026-04-13 00:00:00-04:00,259.859985,260.179993,256.660004,257.825012,5365802,0.0,0.0


## 9. Stock Price Trend Visualisation

The chart below visualises:
- the closing price trend,
- and a 20-day moving average to smooth short-term fluctuations.

In [76]:
hist = hist.copy()
hist["MA20"] = hist["Close"].rolling(window=20).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hist.index,
    y=hist["Close"],
    mode="lines",
    name="Close Price"
))

fig.add_trace(go.Scatter(
    x=hist.index,
    y=hist["MA20"],
    mode="lines",
    name="20-Day Moving Average",
    line=dict(dash="dash")
))

fig.update_layout(
    title=f"{ticker} Closing Price Trend ({period})",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    hovermode="x unified",
    template="plotly_white"
)

import plotly.io as pio
pio.renderers.default = "browser"
fig.show()

## 10. Key Financial Indicators

The following indicators are extracted for the selected company:
- Market Capitalization
- Trailing P/E
- Revenue Growth
- Profit Margins
- Current Ratio
- Debt-to-Equity Ratio

In [77]:
company_name = info.get("longName", ticker)
sector = info.get("sector", "N/A")
industry = info.get("industry", "N/A")

market_cap = info.get("marketCap", "N/A")
trailing_pe = info.get("trailingPE", "N/A")
revenue_growth = info.get("revenueGrowth", "N/A")
profit_margins = info.get("profitMargins", "N/A")
current_ratio = info.get("currentRatio", "N/A")
debt_to_equity = info.get("debtToEquity", "N/A")

financial_indicators_df = pd.DataFrame({
    "Indicator": [
        "Company Name", "Sector", "Industry", "Market Cap",
        "Trailing P/E", "Revenue Growth", "Profit Margins",
        "Current Ratio", "Debt to Equity"
    ],
    "Value": [
        company_name,
        sector,
        industry,
        format_value(market_cap),
        format_value(trailing_pe),
        format_value(revenue_growth, percentage=True),
        format_value(profit_margins, percentage=True),
        format_value(current_ratio),
        format_value(debt_to_equity)
    ]
})

display(financial_indicators_df)

,Indicator,Value
0,Company Name,Apple Inc.
1,Sector,Technology
2,Industry,Consumer Electronics
3,Market Cap,3.79T
4,Trailing P/E,32.68
5,Revenue Growth,15.70%
6,Profit Margins,27.04%
7,Current Ratio,0.97
8,Debt to Equity,102.63


## 11. Financial Dimension Evaluation

A simple rule-based interpretation is applied to four dimensions:
- Liquidity
- Solvency
- Profitability
- Growth

This is intentionally simplified for educational use.

In [78]:
liquidity_msg = evaluate_liquidity(current_ratio)
solvency_msg = evaluate_solvency(debt_to_equity)
profitability_msg = evaluate_profitability(profit_margins)
growth_msg = evaluate_growth(revenue_growth)

evaluation_df = pd.DataFrame({
    "Dimension": ["Liquidity", "Solvency", "Profitability", "Growth"],
    "Interpretation": [liquidity_msg, solvency_msg, profitability_msg, growth_msg]
})

display(evaluation_df)

,Dimension,Interpretation
0,Liquidity,Weak liquidity position.
1,Solvency,Moderate leverage.
2,Profitability,Company shows strong profitability.
3,Growth,Revenue is growing strongly.


## 12. Overall Financial Health Score

The project uses a simple rule-based scoring framework:
- Liquidity: up to 25 points
- Solvency: up to 25 points
- Profitability: up to 25 points
- Growth: up to 25 points

Total maximum score = 100

In [79]:
overall_score, dimension_scores = calculate_overall_score(
    current_ratio,
    debt_to_equity,
    profit_margins,
    revenue_growth
)

score_df = pd.DataFrame({
    "Dimension": ["Liquidity", "Solvency", "Profitability", "Growth", "Overall Rating"],
    "Score / Result": [
        f"{dimension_scores['Liquidity'] if dimension_scores['Liquidity'] is not None else 'N/A'} / 25",
        f"{dimension_scores['Solvency'] if dimension_scores['Solvency'] is not None else 'N/A'} / 25",
        f"{dimension_scores['Profitability'] if dimension_scores['Profitability'] is not None else 'N/A'} / 25",
        f"{dimension_scores['Growth'] if dimension_scores['Growth'] is not None else 'N/A'} / 25",
        score_label(overall_score)
    ]
})

print("Overall Score:", f"{overall_score}/100" if overall_score is not None else "N/A")
display(score_df)

Overall Score: 78/100


,Dimension,Score / Result
0,Liquidity,10 / 25
1,Solvency,18 / 25
2,Profitability,25 / 25
3,Growth,25 / 25
4,Overall Rating,Good


## 13. Summary Insight

In [80]:
summary_text = build_overall_summary(
    liquidity_msg,
    solvency_msg,
    profitability_msg,
    growth_msg,
    overall_score
)

print(summary_text)

Overall summary: The company receives an overall financial health score of 78/100, which is classified as good. Weak liquidity position. Moderate leverage. Company shows strong profitability. Revenue is growing strongly.


## 14. Final Export Table

The final result table mirrors the export function of the Streamlit app.

In [81]:
export_df = pd.DataFrame({
    "Ticker": [ticker],
    "Company Name": [company_name],
    "Sector": [sector],
    "Industry": [industry],
    "Latest Close Price": [latest_close],
    "Price Change": [price_change],
    "Price Change %": [price_change_pct],
    "Average Volume": [avg_volume],
    "Market Cap": [market_cap],
    "Trailing PE": [trailing_pe],
    "Revenue Growth": [revenue_growth],
    "Profit Margins": [profit_margins],
    "Current Ratio": [current_ratio],
    "Debt to Equity": [debt_to_equity],
    "Overall Score": [overall_score],
    "Overall Rating": [score_label(overall_score)],
    "Price Data Source": [price_source],
    "Financial Info Source": [info_source]
})

display(export_df)

,Ticker,Company Name,Sector,Industry,Latest Close Price,Price Change,Price Change %,Average Volume,Market Cap,Trailing PE,Revenue Growth,Profit Margins,Current Ratio,Debt to Equity,Overall Score,Overall Rating,Price Data Source,Financial Info Source
0,AAPL,Apple Inc.,Technology,Consumer Electronics,257.825012,10.636154,4.302845,4.594314e+07,3789492846592,32.67744,0.157,0.27037,0.974,102.63,78,Good,Yahoo Finance,Yahoo Finance


In [82]:
output_file = f"{ticker}_analysis_result.csv"
export_df.to_csv(output_file, index=False)
print(f"Exported result to {output_file}")

Exported result to AAPL_analysis_result.csv


## 15. Interpretation and Product Value

This notebook supports the Streamlit application by showing the complete analytical workflow behind the interactive tool.

### Main Outputs
The project produces:
- a recent stock trend overview,
- selected financial indicators,
- four rule-based financial dimension evaluations,
- an overall financial health score,
- a written summary,
- and an exportable CSV result.

### Value for Users
For beginner users, this tool provides a simple and interpretable way to explore a company’s recent stock and financial condition without navigating a large financial platform.

## 16. Limitations and Possible Improvements

### Limitations
- The scoring model is simplified and rule-based rather than statistically validated.
- Only a limited number of financial indicators are included.
- Yahoo Finance data availability may vary.
- Local fallback files need manual maintenance.
- The analysis is intended for educational use and should not be treated as professional investment advice.

### Possible Improvements
- Add more valuation and performance indicators.
- Include peer comparison between multiple companies.
- Support sector benchmarking.
- Improve the scoring model with more robust financial logic.
- Deploy the app online for easier user access.